## Imports

In [58]:
import numpy as np
import os
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
import time

cwd = Path.cwd()

if "notebooks" in cwd.parts:
    project_root = cwd.parents[cwd.parts[::-1].index("notebooks")]
    os.chdir(project_root)

print("Current working directory:", os.getcwd())

Current working directory: c:\Users\Darío\Desktop\OTROS\vector-recommender


### Load ratings table

In [35]:
RATINGS_PATH = "data/processed/letterboxd_movie_ratings_dataset/ratings.csv"

ratings_df = pd.read_csv(
    RATINGS_PATH,
    engine="python",
)

ratings_df.head()

,_id,movie_id,rating_val,user_id
0,5fc57c5d6758f6963451a07f,feast-2014,7,deathproof
1,5fc57c5d6758f6963451a063,loving-2016,7,deathproof
2,5fc57c5d6758f6963451a0ef,scripted-content,7,deathproof
3,5fc57c5d6758f6963451a060,the-future,4,deathproof
4,5fc57c5c6758f69634519398,mank,5,deathproof


### Filtering Users by Minimum Rating History

For the user-user KNN model, we need enough rating history for each user to obtain a meaningful representation of their preferences and identify similar users.

Based on the previous analysis, we require each user to have rated at least **20 movies**. This threshold provides a sufficiently rich user profile while retaining the vast majority of users in the dataset.

Users with fewer than 20 ratings are therefore excluded from the KNN experiment. This filtering is applied specifically to the modeling dataset and does not modify the original ratings data.

In [36]:
ratings_per_user = (
    ratings_df
    .groupby("user_id")
    .size()
    .sort_values()
)

ratings_per_user.describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count     7477.000000
mean      1481.632607
std       1993.538356
min          1.000000
1%           6.760000
5%          51.000000
10%        124.000000
25%        403.000000
50%        984.000000
75%       1953.000000
90%       3305.000000
95%       4472.000000
99%       7569.200000
max      88289.000000
dtype: float64

In [37]:
thresholds = [1, 2, 5, 10, 20, 50, 100]

for threshold in thresholds:
    n_users = (ratings_per_user >= threshold).sum()
    print(
        f">= {threshold:3} ratings: "
        f"{n_users:,} users"
    )

>=   1 ratings: 7,477 users
>=   2 ratings: 7,451 users
>=   5 ratings: 7,420 users
>=  10 ratings: 7,378 users
>=  20 ratings: 7,301 users
>=  50 ratings: 7,109 users
>= 100 ratings: 6,852 users


In [38]:
MIN_RATINGS_PER_USER = 20

ratings_per_user = ratings_df.groupby("user_id").size()

eligible_users = ratings_per_user[
    ratings_per_user >= MIN_RATINGS_PER_USER
].index

knn_ratings_df = ratings_df[
    ratings_df["user_id"].isin(eligible_users)
].copy()

print(f"Original ratings: {len(ratings_df):,}")
print(f"Filtered ratings: {len(knn_ratings_df):,}")
print(f"Eligible users: {len(eligible_users):,}")

knn_ratings_df.groupby("user_id").size().min()

Original ratings: 11,078,167
Filtered ratings: 11,076,665
Eligible users: 7,301


np.int64(20)

### Train-Test Split

To evaluate the user-user KNN model, the available ratings are divided into **training** and **test** sets.

The split is performed **independently for each user**, rather than randomly across the entire ratings dataset. For each user, 80% of their ratings are assigned to the training set and the remaining 20% to the test set.

The training ratings will be used to build each user's rating profile and identify similar users. The ratings in the test set are hidden from the model and will later be used to evaluate how accurately the KNN model can predict ratings that the users have actually given.

A fixed random seed is used to ensure that the same train-test split can be reproduced across different runs.

In [39]:
train_parts = []
test_parts = []

for user_id, user_ratings in knn_ratings_df.groupby("user_id"):
    train_user, test_user = train_test_split(
        user_ratings,
        test_size=0.2,
        random_state=42,
    )

    train_parts.append(train_user)
    test_parts.append(test_user)

train_df = pd.concat(
    train_parts,
    ignore_index=True,
)

test_df = pd.concat(
    test_parts,
    ignore_index=True,
)

print(f"Train ratings: {len(train_df):,}")
print(f"Test ratings: {len(test_df):,}")

Train ratings: 8,858,420
Test ratings: 2,218,245


In [40]:
train_counts = train_df.groupby("user_id").size()

print(
    f"Minimum ratings per user in train: "
    f"{train_counts.min()}"
)

Minimum ratings per user in train: 16


### User–Movie Interaction Matrix

To prepare the data for collaborative filtering, we transform the ratings dataset into a **user–movie interaction matrix**.

Each row represents a user (`user_id`), each column represents a movie (`movie_id`), and each cell contains the rating given by that user to that movie (`rating_val`).

If a user has not rated a particular movie, the corresponding cell remains missing. This sparse matrix provides the representation required by user-based collaborative filtering methods, allowing us to compare users according to their rating patterns.

It's important to calculate this matrix only whit the trainig data, preventing data leakage

In [41]:
user_movie_matrix = train_df.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating_val",
)

print(user_movie_matrix.shape)
user_movie_matrix.head()

(7301, 265346)


movie_id,0-uhr-15-zimmer-9,00-00,00-08,00-schneider-im-wendekreis-der-eidechse,00-schneider-jagd-auf-nihil-baxter,001,001-2018,001-ing,002-operation-moon-1965,005,...,zyuden-sentai-kyoryuger-100-years-after,zyuden-sentai-kyoryuger-vs-go-busters-the-great-dinosaur-war,zyzzyx-road,zz-top-double-down-live-2009,zz-top-greatest-hits,zz-top-live-at-bonnaroo-2013,zz-top-live-at-montreux-2013,zz-top-live-from-texas,zz-top-live-in-germany-1980,zz-top-that-little-ol-band-from-texas
user_id,,,,,,,,,,,,,,,,,,,,,
007filmreviwer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
007hertzrumble,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0o0o0o0o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11122001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
127gbh,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Finding Similar Users

For the user-user KNN baseline, we identify the **k users most similar to each target user** based on their rating patterns.

We use **cosine similarity** to compare users, considering only the movies that both users have rated. Missing values are not treated as zero, since a missing rating means that the user has not rated the movie rather than assigning it a zero rating.

To avoid unreliable similarities based on very few common movies, we require users to have rated at least **5 common movies**. As an initial baseline, we use **k = 30** nearest neighbors. This value will be kept fixed for the initial evaluation and can be tuned later if necessary.

In [42]:
# Ratings: missing values are represented by 0
ratings_sparse = csr_matrix(
    user_movie_matrix.fillna(0).to_numpy(dtype=np.float32)
)

# Binary matrix: 1 if the user rated the movie, 0 otherwise
rated_sparse = ratings_sparse.copy()
rated_sparse.data = np.ones_like(rated_sparse.data)

In [43]:
def find_similar_users_optimised(
    user_id,
    user_movie_matrix,
    ratings_sparse,
    rated_sparse,
    k=30,
    min_common_items=5,
):
    """
    Find the k most similar users to a given user.

    Similarity is computed using cosine similarity over the movies
    rated by both users.

    Parameters
    ----------
    user_id : str
        ID of the target user.

    user_movie_matrix : pd.DataFrame
        User-movie rating matrix.

    ratings_sparse : scipy.sparse.csr_matrix
        Sparse user-movie rating matrix.

    rated_sparse : scipy.sparse.csr_matrix
        Sparse binary matrix indicating whether a rating exists.

    k : int, default=30
        Number of similar users to return.

    min_common_items : int, default=5
        Minimum number of commonly rated movies required.

    Returns
    -------
    pd.DataFrame
        DataFrame containing the most similar users and their
        cosine similarity.
    """

    if user_id not in user_movie_matrix.index:
        raise ValueError(f"User not found: {user_id}")

    user_idx = user_movie_matrix.index.get_loc(user_id)

    target_ratings = ratings_sparse[user_idx]
    target_mask = rated_sparse[user_idx]

    # ---------------------------------------------------------
    # Number of movies rated by both users
    # ---------------------------------------------------------

    common_items = (
        rated_sparse @ target_mask.T
    ).toarray().ravel()

    # ---------------------------------------------------------
    # Only keep users with enough common movies
    # ---------------------------------------------------------

    valid_mask = common_items >= min_common_items
    valid_mask[user_idx] = False

    valid_indices = np.flatnonzero(valid_mask)

    if len(valid_indices) == 0:
        return pd.DataFrame(
            columns=[
                "user_id",
                "similarity",
                "common_items",
            ]
        )

    # ---------------------------------------------------------
    # Dot product over common movies
    # ---------------------------------------------------------

    dot_products = (
        ratings_sparse @ target_ratings.T
    ).toarray().ravel()

    # ---------------------------------------------------------
    # Norm of target user restricted to common movies
    #
    # For each candidate v:
    #
    # ||u||² = sum_{i in I_uv} r_ui²
    # ---------------------------------------------------------

    target_squared = target_ratings.multiply(target_ratings)

    target_common_squared = (
        rated_sparse @ target_squared.T
    ).toarray().ravel()

    # ---------------------------------------------------------
    # Norm of each candidate user restricted to common movies
    #
    # ||v||² = sum_{i in I_uv} r_vi²
    # ---------------------------------------------------------

    ratings_squared = ratings_sparse.multiply(
        ratings_sparse
    )

    candidate_common_squared = (
        ratings_squared @ target_mask.T
    ).toarray().ravel()

    # ---------------------------------------------------------
    # Cosine similarity
    # ---------------------------------------------------------

    denominator = np.sqrt(
        target_common_squared
        * candidate_common_squared
    )

    similarities = np.zeros_like(dot_products)

    valid_denominator = denominator > 0

    similarities[valid_denominator] = (
        dot_products[valid_denominator]
        / denominator[valid_denominator]
    )

    # ---------------------------------------------------------
    # Build result
    # ---------------------------------------------------------

    similarities_df = pd.DataFrame(
        {
            "user_id": user_movie_matrix.index[valid_indices],
            "similarity": similarities[valid_indices],
            "common_items": common_items[valid_indices].astype(int),
        }
    )

    return (
        similarities_df
        .sort_values("similarity", ascending=False)
        .head(k)
        .reset_index(drop=True)
    )

Test function

In [46]:
user_id = train_df["user_id"].iloc[0]

similar_users = find_similar_users_optimised(
    user_id,
    user_movie_matrix,
    ratings_sparse,
    rated_sparse,
    k=30,
    min_common_items=5
)

similar_users

,user_id,similarity,common_items
0,hhhhhenryc,0.998813,9
1,scrappydoo666,0.998394,7
2,rockypeterson,0.997717,37
3,ldiwald,0.997408,18
4,_yeast_,0.996708,5
5,planetclaires,0.996195,9
6,pzng97,0.996093,59
7,unclenugget,0.995701,10
8,baradwaj_rangan,0.995658,5
9,liquidcomedian,0.995504,30


### Movie Rating Prediction

Once the most similar users have been identified, we use their ratings to generate predictions for movies that the target user has **not rated in the training set**.

For each candidate movie, we consider only the selected neighbors who have rated it. A movie must have been rated by at least **3 neighbors** to be considered, providing a minimum amount of collaborative evidence.

The predicted rating is calculated as a **similarity-weighted average** of the neighbors' ratings. Ratings from users with a higher similarity to the target user therefore have a greater influence on the prediction.

In addition to the predicted rating, we keep the following information:

- `num_neighbors`: number of selected neighbors who rated the movie.
- `similarity_sum`: sum of their similarity scores.

These additional values allow us to inspect how much collaborative evidence supports each prediction, even though the initial recommendations are ordered only by the predicted rating.

The predicted rating for a user $u$ and movie $i$ is computed as a similarity-weighted average of the ratings given by the user's neighbors who have rated that movie:

$$
\hat{r}_{u,i}
=
\frac{
\displaystyle\sum_{v \in N(u,i)} s_{u,v}\,r_{v,i}
}{
\displaystyle\sum_{v \in N(u,i)} s_{u,v}
}
$$

where:

- $\hat{r}_{u,i}$ is the predicted rating of user $u$ for movie $i$.
- $N(u,i)$ is the set of neighbors of $u$ who have rated movie $i$.
- $s_{u,v}$ is the similarity between users $u$ and $v$.
- $r_{v,i}$ is the rating given by neighbor $v$ to movie $i$.

Thus, ratings from more similar users have a greater influence on the predicted rating.

Finally, movies that the target user has already rated in the training set are excluded from the candidates, since they should not be recommended again.

In [ ]:
def predict_movie_ratings(
    user_id,
    similar_users,
    ratings_df,
    min_neighbors=3,
):
    """
    Predict ratings for movies based on the user's similar users.

    Ratings are computed as a similarity-weighted average of the
    ratings given by the user's neighbors.
    """

    # Movies already rated by the target user
    rated_movies = set(
        ratings_df.loc[
            ratings_df["user_id"] == user_id,
            "movie_id",
        ]
    )

    # Ratings from the selected neighbors
    neighbor_ids = similar_users["user_id"].tolist()

    neighbor_ratings = ratings_df[
        ratings_df["user_id"].isin(neighbor_ids)
    ].copy()

    # Add similarity to each rating
    neighbor_ratings = neighbor_ratings.merge(
        similar_users[["user_id", "similarity"]],
        on="user_id",
        how="inner",
    )

    # Remove movies already rated by the target user
    neighbor_ratings = neighbor_ratings[
        ~neighbor_ratings["movie_id"].isin(rated_movies)
    ]

    # Number of neighbors who rated each movie
    movie_stats = (
        neighbor_ratings
        .groupby("movie_id")
        .agg(
            num_neighbors=("user_id", "nunique"),
            similarity_sum=("similarity", "sum"),
        )
    )

    # Keep only movies rated by at least 3 neighbors
    eligible_movies = movie_stats[
        movie_stats["num_neighbors"] >= min_neighbors
    ].index

    neighbor_ratings = neighbor_ratings[
        neighbor_ratings["movie_id"].isin(eligible_movies)
    ]

    # Weighted contribution of each rating
    neighbor_ratings["weighted_rating"] = (
        neighbor_ratings["rating_val"]
        * neighbor_ratings["similarity"]
    )

    predictions = (
        neighbor_ratings
        .groupby("movie_id")
        .agg(
            weighted_rating_sum=("weighted_rating", "sum"),
            similarity_sum=("similarity", "sum"),
            num_neighbors=("user_id", "nunique"),
        )
    )

    # Similarity-weighted average
    predictions["predicted_rating"] = (
        predictions["weighted_rating_sum"]
        / predictions["similarity_sum"]
    )

    return (
        predictions[
            [
                "predicted_rating",
                "num_neighbors",
                "similarity_sum",
            ]
        ]
        .sort_values(
            "predicted_rating",
            ascending=False,
        )
    )

Test function

In [48]:
predicted_movies = predict_movie_ratings(
    user_id=user_id,
    similar_users=similar_users,
    ratings_df=train_df,
)

predicted_movies.head(20)

,predicted_rating,num_neighbors,similarity_sum
movie_id,,,
certified-copy,10.000000,3,2.980002
yi-yi,10.000000,3,2.981498
the-double-life-of-veronique,10.000000,3,2.986633
magnolia,10.000000,4,3.979074
man-with-a-movie-camera,10.000000,4,3.979294
seven-samurai,10.000000,7,6.952745
tokyo-story,10.000000,3,2.976898
2046,10.000000,3,2.987123
andrei-rublev,10.000000,4,3.978902


### Evaluating Ratings from the Test Set

To evaluate the KNN user-user model, we use the ratings that were held out in the **test set**.

For each movie rated by the target user in the test set, we use the ratings provided by their similar users in the training set to calculate a predicted rating using the same similarity-weighted average defined previously.

Unlike the recommendation stage, we **do not require a minimum number of neighbors** to have rated the movie. If at least one of the selected neighbors has rated a test movie, we can obtain a prediction and use it for evaluation. This allows us to evaluate the model under the actual information available to it.

However, if none of the user's neighbors has rated a particular test movie, the model cannot produce a collaborative prediction for that movie. These ratings are therefore considered **not covered** by the model and are excluded from the rating prediction evaluation.

For each evaluable test rating, we keep:

- `actual_rating`: the user's actual rating in the test set.
- `predicted_rating`: the rating predicted by KNN.
- `num_neighbors`: number of neighbors who rated the movie.
- `similarity_sum`: total similarity of those neighbors.

This will allow us to compare the predicted ratings with the actual ratings and later calculate metrics such as **MAE** and **RMSE**, while also measuring the model's **coverage**.

In [50]:
def predict_test_ratings(
    user_id,
    similar_users,
    train_df,
    test_df,
    min_neighbors=1,
):
    """
    Predict the ratings of the target user's test movies using
    the ratings provided by their similar users.

    Unlike the recommendation stage, the default minimum number
    of neighbors is 1, since any available neighbor provides
    information that can be used to evaluate the model.

    Parameters
    ----------
    user_id : str
        ID of the target user.

    similar_users : pd.DataFrame
        Similar users and their similarity scores.

    train_df : pd.DataFrame
        Training ratings used to generate predictions.

    test_df : pd.DataFrame
        Test ratings containing the actual ratings.

    min_neighbors : int, default=1
        Minimum number of neighbors that must have rated a movie
        for it to be included in the evaluation.

    Returns
    -------
    pd.DataFrame
        Predicted and actual ratings for the user's test movies
        that can be predicted.
    """

    # Movies in the target user's test set
    user_test = test_df[
        test_df["user_id"] == user_id
    ].copy()

    if user_test.empty:
        return pd.DataFrame()

    test_movies = set(user_test["movie_id"])

    # Selected neighbors
    neighbor_ids = similar_users["user_id"].tolist()

    # Ratings given by those neighbors
    neighbor_ratings = train_df[
        train_df["user_id"].isin(neighbor_ids)
        & train_df["movie_id"].isin(test_movies)
    ].copy()

    if neighbor_ratings.empty:
        return pd.DataFrame()

    # Add similarity to each rating
    neighbor_ratings = neighbor_ratings.merge(
        similar_users[["user_id", "similarity"]],
        on="user_id",
        how="inner",
    )

    # Number of neighbors who rated each movie
    movie_stats = (
        neighbor_ratings
        .groupby("movie_id")
        .agg(
            num_neighbors=("user_id", "nunique"),
            similarity_sum=("similarity", "sum"),
        )
    )

    # Keep only movies with enough neighbors
    eligible_movies = movie_stats[
        movie_stats["num_neighbors"] >= min_neighbors
    ].index

    neighbor_ratings = neighbor_ratings[
        neighbor_ratings["movie_id"].isin(eligible_movies)
    ].copy()

    # Weighted contribution of each rating
    neighbor_ratings["weighted_rating"] = (
        neighbor_ratings["rating_val"]
        * neighbor_ratings["similarity"]
    )

    # Aggregate the information for each movie
    predictions = (
        neighbor_ratings
        .groupby("movie_id")
        .agg(
            weighted_rating_sum=("weighted_rating", "sum"),
            similarity_sum=("similarity", "sum"),
            num_neighbors=("user_id", "nunique"),
        )
    )

    # Similarity-weighted average
    predictions["predicted_rating"] = (
        predictions["weighted_rating_sum"]
        / predictions["similarity_sum"]
    )

    # Add the actual test rating
    predictions = predictions.merge(
        user_test[["movie_id", "rating_val"]].rename(
            columns={"rating_val": "actual_rating"}
        ),
        left_index=True,
        right_on="movie_id",
        how="inner",
    )

    return predictions[
        [
            "movie_id",
            "predicted_rating",
            "actual_rating",
            "num_neighbors",
            "similarity_sum",
        ]
    ].set_index("movie_id")

In [51]:
test_predictions = predict_test_ratings(
    user_id=user_id,
    similar_users=similar_users,
    train_df=train_df,
    test_df=test_df,
)

test_predictions

,predicted_rating,actual_rating,num_neighbors,similarity_sum
movie_id,,,,
20th-century-women,8.000000,7,1,0.992897
28-days-later,9.000000,8,1,0.992897
a-beautiful-mind,8.000000,4,1,0.991602
a-clockwork-orange,9.833688,10,6,5.965883
a-ghost-story-2017,8.000000,8,1,0.993803
...,...,...,...,...
unbreakable,9.000000,10,1,0.992897
uncut-gems,9.334068,10,6,5.964920
watchmen,9.000000,10,1,0.992897


Results and metrics for one user

In [52]:
# Number of test ratings that could potentially be predicted
total_test_ratings = len(
    test_df[test_df["user_id"] == user_id]
)

# Number of test ratings for which KNN produced a prediction
covered_test_ratings = len(test_predictions)

# Coverage
coverage = (
    covered_test_ratings / total_test_ratings
    if total_test_ratings > 0
    else 0
)

# Rating prediction errors
mae = mean_absolute_error(
    test_predictions["actual_rating"],
    test_predictions["predicted_rating"],
)

rmse = root_mean_squared_error(
    test_predictions["actual_rating"],
    test_predictions["predicted_rating"],
)

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Coverage: {coverage:.2%}")

MAE: 1.0669
RMSE: 1.5500
Coverage: 32.74%


## Results for all users

Firstly we perform a test for 50 users to estimate the time that it takes to make the predictions with our actual functions, that are unoptimised.

In [53]:
sample_users = test_df["user_id"].unique()[:50]

start_time = time.time()

for user_id in sample_users:

    similar_users = find_similar_users_optimised(
        user_id=user_id,
        user_movie_matrix=user_movie_matrix,
        ratings_sparse=ratings_sparse,
        rated_sparse=rated_sparse,
        k=30,
        min_common_items=5,
    )

    if similar_users.empty:
        continue

    user_predictions = predict_test_ratings(
        user_id=user_id,
        similar_users=similar_users,
        train_df=train_df,
        test_df=test_df,
    )

elapsed = time.time() - start_time

print(f"Time for 50 users: {elapsed:.2f} seconds")
print(f"Estimated time for all users: {elapsed / 50 * len(test_df['user_id'].unique()) / 60:.1f} minutes")

Time for 50 users: 25.00 seconds
Estimated time for all users: 60.8 minutes


In [54]:
all_predictions = []
total_test_ratings = 0

for user_id in test_df["user_id"].unique():

    # Test ratings for this user
    user_test = test_df[test_df["user_id"] == user_id]

    total_test_ratings += len(user_test)

    # Find similar users using only training data
    similar_users = find_similar_users_optimised(
        user_id=user_id,
        user_movie_matrix=user_movie_matrix,
        ratings_sparse=ratings_sparse,
        rated_sparse=rated_sparse,
        k=30,
        min_common_items=5,
    )

    # No neighbors available
    if similar_users.empty:
        continue

    # Predict the user's test ratings
    user_predictions = predict_test_ratings(
        user_id=user_id,
        similar_users=similar_users,
        train_df=train_df,
        test_df=test_df,
    )

    if user_predictions.empty:
        continue

    # Keep user_id for later analysis
    user_predictions = user_predictions.reset_index()
    user_predictions["user_id"] = user_id

    all_predictions.append(user_predictions)

In [59]:
# Combine all predictions
predictions_df = pd.concat(all_predictions, ignore_index=True)

# Keep only the information we need from test
test_eval = test_df[["user_id", "movie_id", "rating_val"]].copy()

# Merge real ratings with predictions
evaluation_df = test_eval.merge(
    predictions_df[["user_id", "movie_id", "predicted_rating"]],
    on=["user_id", "movie_id"],
    how="left",
)

# --------------------------------------------------
# COVERAGE
# --------------------------------------------------

total_ratings = len(evaluation_df)
predicted_ratings = evaluation_df["predicted_rating"].notna().sum()
unpredicted_ratings = total_ratings - predicted_ratings

rating_coverage = predicted_ratings / total_ratings

total_users = evaluation_df["user_id"].nunique()
users_with_predictions = (
    evaluation_df.loc[
        evaluation_df["predicted_rating"].notna(),
        "user_id"
    ].nunique()
)

user_coverage = users_with_predictions / total_users

print("=== COVERAGE ===")
print(f"Total test ratings:       {total_ratings:,}")
print(f"Predicted ratings:        {predicted_ratings:,}")
print(f"Unpredicted ratings:      {unpredicted_ratings:,}")
print(f"Rating coverage:          {rating_coverage:.2%}")
print()
print(f"Total test users:         {total_users:,}")
print(f"Users with predictions:   {users_with_predictions:,}")
print(f"Users without predictions:{total_users - users_with_predictions:,}")
print(f"User coverage:            {user_coverage:.2%}")


# --------------------------------------------------
# ACCURACY METRICS
# --------------------------------------------------

# Only evaluate ratings for which we have a prediction
evaluated_df = evaluation_df.dropna(subset=["predicted_rating"]).copy()

y_true = evaluated_df["rating_val"]
y_pred = evaluated_df["predicted_rating"]

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print()
print("=== PREDICTION ACCURACY ===")
print(f"Ratings evaluated:        {len(evaluated_df):,}")
print(f"MAE:                      {mae:.4f}")
print(f"RMSE:                     {rmse:.4f}")

=== COVERAGE ===
Total test ratings:       2,218,245
Predicted ratings:        481,610
Unpredicted ratings:      1,736,635
Rating coverage:          21.71%

Total test users:         7,301
Users with predictions:   7,301
Users without predictions:0
User coverage:            100.00%

=== PREDICTION ACCURACY ===
Ratings evaluated:        481,610
MAE:                      1.2780
RMSE:                     1.8771
